## Importing necessary libraries & loading the cleaned data

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [9]:
df = pd.read_csv("../data/processed/processed_railway.csv")
df

,ohe__Purchase Type_Online,ohe__Purchase Type_Station,ohe__Payment Method_Contactless,ohe__Payment Method_Credit Card,ohe__Payment Method_Debit Card,ohe__Railcard_Adult,ohe__Railcard_Disabled,ohe__Railcard_No Railcard,ohe__Railcard_Senior,ohe__Ticket Type_Advance,...,ohe__Journey Status_On Time,label__Departure Station,label__Arrival Destination,label__Refund Request,remainder__Price,remainder__Is Weekend,remainder__Is Night Departure,remainder__Lead Hours,remainder__Delay Minutes,remainder__Duration Minutes
0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,6.0,12.0,0.0,3.784190,0.0,0.0,6.354915,0.000000,150.0
1,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,5.0,31.0,0.0,3.178054,0.0,0.0,5.948991,2.236068,115.0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,3.0,18.0,0.0,1.386294,0.0,0.0,5.815301,0.000000,30.0
3,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,6.0,23.0,0.0,2.639057,0.0,0.0,5.661188,0.000000,60.0
4,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,3.0,13.0,0.0,4.343805,0.0,0.0,4.782209,0.000000,135.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31648,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,8.0,12.0,0.0,1.609438,0.0,0.0,0.825880,0.000000,30.0
31649,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,4.0,0.0,0.0,2.397895,0.0,0.0,0.908483,0.000000,80.0
31650,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,8.0,12.0,0.0,1.609438,0.0,0.0,0.835201,0.000000,30.0
31651,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,4.0,0.0,0.0,2.397895,0.0,0.0,0.870014,0.000000,80.0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 30 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   ohe__Purchase Type_Online                 31653 non-null  float64
 1   ohe__Purchase Type_Station                31653 non-null  float64
 2   ohe__Payment Method_Contactless           31653 non-null  float64
 3   ohe__Payment Method_Credit Card           31653 non-null  float64
 4   ohe__Payment Method_Debit Card            31653 non-null  float64
 5   ohe__Railcard_Adult                       31653 non-null  float64
 6   ohe__Railcard_Disabled                    31653 non-null  float64
 7   ohe__Railcard_No Railcard                 31653 non-null  float64
 8   ohe__Railcard_Senior                      31653 non-null  float64
 9   ohe__Ticket Type_Advance                  31653 non-null  float64
 10  ohe__Ticket Type_Anytime          

## Splitting

In [ ]:
X = df.drop(columns=["label__Refund Request"])
y = df["label__Refund Request"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [20]:
imputer = SimpleImputer(strategy="median")
X_train_clean = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)

smote = SMOTE(sampling_strategy=0.3, random_state=42)
X_train, y_train = smote.fit_resample(X_train_clean, y_train)

## Model Training and evaluation

### Decision Tree

In [23]:
dt_pipeline = Pipeline([
    ('clf', DecisionTreeClassifier(random_state=42))
])

dt_param_grid = {
    'clf__criterion': ['gini', 'entropy'],
    'clf__max_depth': [None, 3, 5, 10, 15],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4]
}

dt_grid_search = GridSearchCV(estimator=dt_pipeline, param_grid=dt_param_grid, cv=5, scoring='f1', n_jobs=-1)
dt_grid_search.fit(X_train, y_train)

best_model = dt_grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("--- Decision Tree Classifier ---")
print(f"Best Parameters: {dt_grid_search.best_params_}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

--- Decision Tree Classifier ---
Best Parameters: {'clf__criterion': 'gini', 'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 2}
Accuracy: 97.38%
              precision    recall  f1-score   support

         0.0       0.99      0.98      0.99      6123
         1.0       0.59      0.66      0.62       208

    accuracy                           0.97      6331
   macro avg       0.79      0.82      0.80      6331
weighted avg       0.98      0.97      0.97      6331



### Random Forest Classifier (RFC)

In [ ]:
rf_pipeline = Pipeline([
    ('clf', RandomForestClassifier(random_state=42))
])

rf_param_grid = {
    'clf__n_estimators': [50, 100, 200, 500],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_split': [2, 5]
}

rf_grid_search = GridSearchCV(estimator=rf_pipeline, param_grid=rf_param_grid, cv=5, scoring='f1', n_jobs=-1)
rf_grid_search.fit(X_train, y_train)

best_model = rf_grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("--- Random Forest Classifier ---")
print(f"Best Parameters: {rf_grid_search.best_params_}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

--- Random Forest Classifier ---
Best Parameters: {'clf__max_depth': 20, 'clf__min_samples_split': 5, 'clf__n_estimators': 500}
Accuracy: 97.63%
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      6123
         1.0       0.65      0.62      0.63       208

    accuracy                           0.98      6331
   macro avg       0.82      0.80      0.81      6331
weighted avg       0.98      0.98      0.98      6331



### XGBoost Classifier

In [28]:
xgboost_pipeline = Pipeline([
    ('clf', XGBClassifier(random_state=42, eval_metric='logloss'))
])

xgboost_param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__learning_rate': [0.01, 0.1, 0.2],
    'clf__max_depth': [3, 5, 7]
}

xgb_grid_search = GridSearchCV(estimator=xgboost_pipeline, param_grid=xgboost_param_grid, cv=5, scoring='f1', n_jobs=-1)
xgb_grid_search.fit(X_train, y_train)

best_model = xgb_grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("--- XGBoost Classifier ---")
print(f"Best Parameters: {xgb_grid_search.best_params_}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

c:\Users\Nour\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
1 fits failed out of a total of 135.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Nour\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Nour\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\Nour\AppData\Loc

--- XGBoost Classifier ---
Best Parameters: {'clf__learning_rate': 0.2, 'clf__max_depth': 7, 'clf__n_estimators': 200}
Accuracy: 98.28%
              precision    recall  f1-score   support

         0.0       0.98      1.00      0.99      6123
         1.0       0.91      0.53      0.67       208

    accuracy                           0.98      6331
   macro avg       0.95      0.76      0.83      6331
weighted avg       0.98      0.98      0.98      6331



### LightGBM Classifier

In [31]:
lgb_pipeline = Pipeline([
    ('clf', LGBMClassifier(random_state=42, verbose=-1))
])

lgb_param_grid = {
    'clf__n_estimators': [200, 500, 1000],
    'clf__learning_rate': [0.01, 0.1, 0.2, 0.3],
    'clf__max_depth': [7, 10],
    'clf__num_leaves': [63, 127, 255]
}

lgb_grid_search = GridSearchCV(estimator=lgb_pipeline, param_grid=lgb_param_grid, cv=5, scoring='f1', n_jobs=-1)
lgb_grid_search.fit(X_train, y_train)

best_model = lgb_grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("--- LightGBM Classifier ---")
print(f"Best Parameters: {lgb_grid_search.best_params_}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))


--- LightGBM Classifier ---
Best Parameters: {'clf__learning_rate': 0.2, 'clf__max_depth': 10, 'clf__n_estimators': 500, 'clf__num_leaves': 255}
Accuracy: 97.87%
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      6123
         1.0       0.72      0.58      0.64       208

    accuracy                           0.98      6331
   macro avg       0.85      0.79      0.82      6331
weighted avg       0.98      0.98      0.98      6331



### CatBoost Classifier

In [32]:
cat_pipeline = Pipeline([
    ('clf', CatBoostClassifier(random_state=42, verbose=0))
])

cat_param_grid = {
    'clf__iterations': [100, 200, 300],
    'clf__learning_rate': [0.01, 0.1, 0.2],
    'clf__depth': [4, 6, 8, 10]
}

cat_grid_search = GridSearchCV(estimator=cat_pipeline, param_grid=cat_param_grid, cv=5, scoring='f1', n_jobs=-1)
cat_grid_search.fit(X_train, y_train)

best_model = cat_grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("--- CatBoost Classifier ---")
print(f"Best Parameters: {cat_grid_search.best_params_}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

--- CatBoost Classifier ---
Best Parameters: {'clf__depth': 10, 'clf__iterations': 100, 'clf__learning_rate': 0.2}
Accuracy: 97.79%
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      6123
         1.0       0.67      0.64      0.66       208

    accuracy                           0.98      6331
   macro avg       0.83      0.81      0.82      6331
weighted avg       0.98      0.98      0.98      6331



### Benchmark Evaluation

In [33]:
trained_grid_searches = {
    "Decision Tree": dt_grid_search,
    "Random Forest": rf_grid_search,
    "XGBoost": xgb_grid_search,
    "LightGBM": lgb_grid_search,
    "CatBoost": cat_grid_search
}

benchmark_results = []

for name, grid_search in trained_grid_searches.items():
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    benchmark_results.append({
        "Model": name,
        "Test Accuracy (%)": round(accuracy_score(y_test, y_pred) * 100, 2),
        "F1 Score (%)": round(f1_score(y_test, y_pred, average='weighted') * 100, 2),
        "Precision (%)": round(precision_score(y_test, y_pred, average='weighted') * 100, 2),
        "Recall (%)": round(recall_score(y_test, y_pred, average='weighted') * 100, 2),
        "Recall Class 1 (%)": round(recall_score(y_test, y_pred, pos_label=1) * 100, 2),
        "Precision Class 1 (%)": round(precision_score(y_test, y_pred, pos_label=1) * 100, 2),
        "Best Params": str(grid_search.best_params_)
    })

benchmark_df = pd.DataFrame(benchmark_results).sort_values(
    by=["Test Accuracy (%)", "F1 Score (%)"], 
    ascending=False
).reset_index(drop=True)

benchmark_df

,Model,Test Accuracy (%),F1 Score (%),Precision (%),Recall (%),Recall Class 1 (%),Precision Class 1 (%),Best Params
0,XGBoost,98.28,98.06,98.18,98.28,52.88,90.91,"{'clf__learning_rate': 0.2, 'clf__max_depth': ..."
1,LightGBM,97.87,97.76,97.70,97.87,58.17,71.60,"{'clf__learning_rate': 0.2, 'clf__max_depth': ..."
2,CatBoost,97.79,97.76,97.74,97.79,63.94,67.17,"{'clf__depth': 10, 'clf__iterations': 100, 'cl..."
3,Random Forest,97.63,97.61,97.59,97.63,62.02,64.50,"{'clf__max_depth': 20, 'clf__min_samples_split..."
4,Decision Tree,97.38,97.45,97.53,97.38,65.87,59.05,"{'clf__criterion': 'gini', 'clf__max_depth': N..."


CatBoost is the winner